# LightGBM with Optuna (Native NaN Handling)
This notebook trains a LightGBM model utilizing Optuna for hyperparameter tuning. It uses LightGBM's native handling of missing values (NaN) to optimize performance.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from lightgbm import LGBMClassifier
import optuna
import os
import sklearn

In [ ]:
sklearn.set_config(transform_output="pandas")

In [ ]:
# Load data - using train_original.csv to get the native NaNs
train = pd.read_csv('../datasets/train_original.csv')
test = pd.read_csv('../datasets/test.csv')

In [ ]:
X = train.drop(['id', 'addicted_label'], axis=1)
y = train['addicted_label']
X_test = test.drop(['id'], axis=1)

In [ ]:
# Proper Ordinal Mappings
stress_mapping = {'Low': 0, 'Medium': 1, 'High': 2, 'Unknown': -1}
impact_mapping = {'No': 0, 'Yes': 1, 'Unknown': -1}

In [ ]:
def apply_mappings(df):
    df_out = df.copy()
    # Map ordinals, fill missing in these specific columns with 'Unknown' mapping (-1) if any
    df_out['stress_level'] = df_out['stress_level'].map(stress_mapping).fillna(-1).astype(int)
    df_out['academic_work_impact'] = df_out['academic_work_impact'].map(impact_mapping).fillna(-1).astype(int)
    
    # Leave gender as category
    df_out['gender'] = df_out['gender'].fillna('Unknown').astype('category')
    
    return df_out

In [ ]:
X_preprocessed = apply_mappings(X)
X_test_preprocessed = apply_mappings(X_test)

In [ ]:
# Feature Engineering
denom_screen = X_preprocessed['daily_screen_time_hours'].replace(0, 0.001)
denom_notif = X_preprocessed['notifications_per_day'].replace(0, 0.001)
X_preprocessed['social_media_ratio'] = X_preprocessed['social_media_hours'] / denom_screen
X_preprocessed['gaming_ratio'] = X_preprocessed['gaming_hours'] / denom_screen
X_preprocessed['work_study_ratio'] = X_preprocessed['work_study_hours'] / denom_screen
X_preprocessed['app_opens_per_hour'] = X_preprocessed['app_opens_per_day'] / denom_screen
X_preprocessed['notifications_to_opens_ratio'] = X_preprocessed['app_opens_per_day'] / denom_notif
X_preprocessed['sleep_deficit'] = 8.0 - X_preprocessed['sleep_hours']

In [ ]:
denom_screen_test = X_test_preprocessed['daily_screen_time_hours'].replace(0, 0.001)
denom_notif_test = X_test_preprocessed['notifications_per_day'].replace(0, 0.001)
X_test_preprocessed['social_media_ratio'] = X_test_preprocessed['social_media_hours'] / denom_screen_test
X_test_preprocessed['gaming_ratio'] = X_test_preprocessed['gaming_hours'] / denom_screen_test
X_test_preprocessed['work_study_ratio'] = X_test_preprocessed['work_study_hours'] / denom_screen_test
X_test_preprocessed['app_opens_per_hour'] = X_test_preprocessed['app_opens_per_day'] / denom_screen_test
X_test_preprocessed['notifications_to_opens_ratio'] = X_test_preprocessed['app_opens_per_day'] / denom_notif_test
X_test_preprocessed['sleep_deficit'] = 8.0 - X_test_preprocessed['sleep_hours']

In [ ]:
# Optuna Hyperparameter Tuning
def objective(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 200, 600),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
        'num_leaves': trial.suggest_int('num_leaves', 31, 150),
        'max_depth': trial.suggest_int('max_depth', 4, 12),
        'min_child_samples': trial.suggest_int('min_child_samples', 20, 100),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'is_unbalance': True,
        'random_state': 42,
        'verbose': -1,
        'n_jobs': -1
    }
    
    cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
    scores = []
    
    # We use a subset of data if we want to tune faster, but for best results we use the whole train set.
    # To keep the tuning somewhat fast on 690k rows, we'll just run it.
    for train_idx, val_idx in cv.split(X_preprocessed, y):
        X_tr, y_tr = X_preprocessed.iloc[train_idx], y.iloc[train_idx]
        X_val, y_val = X_preprocessed.iloc[val_idx], y.iloc[val_idx]
        
        model = LGBMClassifier(**params)
        model.fit(X_tr, y_tr)
        
        preds = model.predict_proba(X_val)[:, 1]
        scores.append(roc_auc_score(y_val, preds))
        
    return np.mean(scores)

In [ ]:
print("Starting Optuna tuning (15 trials)...")
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=15)

In [ ]:
print(f"Best ROC-AUC Score: {study.best_value}")
print(f"Best Params: {study.best_params}")

In [ ]:
print("Training final model with best params on ALL data...")
best_params = study.best_params
best_params['random_state'] = 42
best_params['verbose'] = -1
best_params['is_unbalance'] = True
best_params['n_jobs'] = -1

In [ ]:
final_model = LGBMClassifier(**best_params)
final_model.fit(X_preprocessed, y)

In [ ]:
print("Predicting final results...")
final_preds = final_model.predict_proba(X_test_preprocessed)[:, 1]

In [ ]:
os.makedirs('../submissions', exist_ok=True)
submission = pd.DataFrame({'id': test['id'], 'addicted_label': final_preds})
submission.to_csv('../submissions/native_nan_fixed_optuna.csv', index=False)
print("Submission saved to submissions/native_nan_fixed_optuna.csv")